# 手撕 Dropout（Invert Dropout）

## 背景
训练时以概率 p 随机置零，存活部分缩放 1/(1-p) 保持期望不变。
推理时恒等变换。Invert Dropout 在训练时做缩放，推理时无需操作。

## 考察点
- invert dropout 原理（为什么训练时缩放）
- 期望保持性：E[dropout(x)] = x
- training/eval 模式切换

In [ ]:
import torch
import torch.nn as nn

class Dropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        assert 0 <= p < 1
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        mask = (torch.rand_like(x) > self.p).float()
        return x * mask / (1 - self.p)  # invert dropout

In [ ]:
# 验证期望保持性
torch.manual_seed(42)
dropout = Dropout(p=0.3)
dropout.train()
x = torch.ones(10000)
out = dropout(x)
# E[dropout(x)] 应 ≈ 1.0
assert abs(out.mean().item() - 1.0) < 0.05, f"期望应≈1, 实际={out.mean().item()}"
# eval 模式应恒等
dropout.eval()
out_eval = dropout(x)
assert torch.equal(out_eval, x), "eval 模式应恒等"
print(f"训练模式期望: {out.mean().item():.4f} (应≈1.0)")
print(f"训练模式存活率: {(out != 0).float().mean().item():.4f} (应≈0.7)")
print("✅ 期望保持性 + eval 恒等验证通过")